# Test your setup

This notebook checks that your environment is configured correctly — it imports the workshop helpers, loads your `.env`, and runs a tiny agent against your chosen model provider. If the cells below run cleanly, you're ready to start the workshop.

## 1. Pick the right kernel

Before running any cell, you need to point this notebook at the Python environment where your workshop dependencies are installed. Open the kernel picker (top-right of the notebook in VS Code, or **Kernel → Change kernel** in Jupyter) and choose:

| If you set up via... | Pick this kernel |
|---|---|
| **Codespace / Dev Container** | **AI Podcast Studio** (under *Jupyter Kernels*, **not** *Python Environments*) |
| **Local virtual environment** | The Python interpreter inside your `.venv` (e.g. `Python 3.x ('.venv': venv)`) |

> Don't see your venv in the list? In VS Code, run **Python: Select Interpreter** from the Command Palette (`Cmd/Ctrl + Shift + P`) and pick the one in `.venv` first — the kernel picker will then offer it.

## 2. Run the cells below

Run each cell in order. If the import cell fails with `ImportError: cannot import name 'load_env' from 'utils'`, you're either on the wrong kernel or you skipped `pip install -e .` during local setup — go back and check both.

In [1]:
# Set up the environment
from utils import load_env, create_agent, stream_response, AgentOptions
load_env()

True

---
## 1. A simple agent

The simplest agent is just a name and a set of instructions. The instructions are the agent's personality and brief — they decide how it sounds and what it cares about.

Try changing the instructions below to see how the same question gets a very different answer.

In [2]:
simple_agent = create_agent(
    AgentOptions(
        name="Simple Southern Agent",
        instructions="You are my assistant. Answer my questions in a southern accent with southern charm.",
    )
)

In [3]:
simple_response = await stream_response(simple_agent.run("hows the weather in Perth this weekend?", stream=True))

Howdy! I’d love to help, but I don’t have live weather access in this chat—so I can’t say exactly how it’s looking in **Perth this weekend** without the forecast data.

Quick question so I get it right: when you say **Perth**, do you mean **Perth, Western Australia**? If you tell me that (or paste a forecast link/screenshot), I’ll help you break down what to expect—temps, rain chances, and whether it’ll be more “sunny and breezy” or “hold onto your hat” territory.


---
## 2. Giving your agent tools

An agent on its own only knows what's in its training data. Tools let it reach beyond that — call a function, look something up, take an action in the world.

First, see what happens when a news agent has no tools and is asked about today's headlines. Then give it a `get_todays_headlines` tool and watch the response change.

In [4]:
agent_with_no_tools = create_agent(
    AgentOptions(
        name="News Agent",
        instructions="You are a helpful news assistant. Answer questions about current events.",
    )
)

no_tools_response = await stream_response(agent_with_no_tools.run("What's in the news today?", stream=True))

I can help, but I don’t have live news access in this chat. If you tell me your country/region (or a few topics you care about), I can summarize what’s typically in the news and help you find the latest headlines fast—or, if you paste links/headlines you’re seeing, I’ll break them down.

Quick questions:
1) What country/region are you in (US, UK, India, etc.)?
2) Any topics: politics, business, tech, world, sports, markets, climate?


In [5]:
def get_todays_headlines() -> str:
    return """
    1. Global reforestation effort plants one billion trees, smashing annual record
    2. New cancer vaccine shows 90 percent success rate in landmark clinical trial
    3. Remote Australian town powered entirely by renewable energy for first time
    4. Endangered whale population rebounds to highest count in 50 years
    5. Local community garden program credited with cutting food insecurity by half
    """

In [6]:
agent_with_tools = create_agent(
    AgentOptions(
        name="News Agent",
        instructions="You are a helpful news assistant. Answer questions about current events.",
        tools=[get_todays_headlines]
    )
)

tools_response = await stream_response(agent_with_tools.run("What's in the news today?", stream=True))

Here are some headlines in the news today:

1. **Global reforestation effort**: Plans to plant **one billion trees**, breaking the annual record.  
2. **Medical breakthrough**: A **new cancer vaccine** reports a **90% success rate** in a landmark clinical trial.  
3. **Renewable energy milestone**: A **remote Australian town** powered entirely by **renewables** for the first time.  
4. **Wildlife recovery**: An **endangered whale population** rebounds to its **highest count in 50 years**.  
5. **Community impact**: A local **community garden program** is credited with cutting **food insecurity by half**.
